In [ ]:
# Stage 06F: cached Gemini pair diagnosis. No fits or new predictions.
import os
os.environ.update({
    'OMP_NUM_THREADS': '2', 'OPENBLAS_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2',
    'NUMEXPR_NUM_THREADS': '2',
})
import hashlib
import json
import runpy
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path


In [ ]:
# Fixed scope: Stage 06E V2 saved raw-control evidence and official train.
# The same passages recur in three contexts; contexts are not independent data.
# This diagnostic changes no prediction and cannot increase the official score.
STAGE06F_INPUT_VERSION_IDS = [170484, 359514801]
STAGE06F_ROOTS = {
    'sources': Path('/kaggle/working/stage06f_sources'),
    'pipeline': Path('/kaggle/working/stage06f_pipeline'),
    'diagnostics': Path('/kaggle/working/stage06f_diagnostics'),
}
STAGE06F_STARTED = time.monotonic()
STAGE06F_PHASE = 'FRESH_ROOT_PREFLIGHT'
STAGE06F_PIPELINE_CREATED = False


In [ ]:
# Preserve phase and failure receipts; never overwrite a prior trial.
def stage06f_pipeline_json(name, value):
    path = STAGE06F_ROOTS['pipeline'] / name
    text = json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n'
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(text, encoding='utf-8')
    temporary.replace(path)

def stage06f_pipeline_status(phase, **extra):
    global STAGE06F_PHASE
    STAGE06F_PHASE = phase
    value = dict(status=phase, observed_utc=datetime.now(timezone.utc).isoformat(),
                 elapsed_seconds=time.monotonic() - STAGE06F_STARTED, **extra)
    stage06f_pipeline_json('status.json', value)
    print('STAGE06F_PIPELINE_PHASE', json.dumps(value, allow_nan=False), flush=True)


In [ ]:
# These are inert reviewed text literals until verified materialization.
STAGE06F_SOURCE_TEXT = {'stage06f_pair_diagnostics.py': '"""Fixed Stage 06F pair diagnosis: cached probability arithmetic only.\n\nNo estimator, model, regrouping, new prediction, optimization or text output.\nProduction requires immutable E output and train bytes plus a source/plan freeze.\n"""\nimport hashlib\nimport json\nimport os\nimport platform\nimport re\nimport time\nimport unicodedata\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nPAIR = (\'gemini-3.7-flash\', \'gemini-3.8-flash\')\nSCENARIOS = (1, 3, 6)\nMEMBERS = (\'stylegram\', \'lora\', \'lora2\', \'tokattn\', \'tokattn_deb\', \'deb_plain\', \'deb_center\')\nWEIGHTS = dict(zip(MEMBERS, (1.516, .177, .113, .153, .185, .05, .25)))\nEXPECTED_HASHES = {\n    \'train.csv\': \'40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\',\n    \'ensemble_arrays.npz\': \'9348eeed149ad48bdd828d8055e945f57e384d8917c654125c2ebbe8d4b99519\',\n    \'ensemble_result.json\': \'9717bcb12f0c730b4af631067621969dcdbf239e6bed2f84304b470a38458018\',\n    \'output_manifest.json\': \'626b428135f4ff377d593de29c716042a07593c74678893f874c91b7b71e2bbf\',\n    \'raw_control_scenario1_confusion.csv\': \'a48b61f2a7ffb50764d44bb060adb925e13122abd63fed4730c9ba097685c595\',\n    \'raw_control_scenario3_confusion.csv\': \'fae2720b0f34b0bb701bfdffaace99051f0add84d52c91e6973aa6f124299463\',\n    \'raw_control_scenario6_confusion.csv\': \'938f3bf5a1ea51fdda8b8a75512810c5c74002063e741feef0d8e5dd614c7513\',\n}\nCANONICAL_SHA = \'2c6aa02a0bafd9389f436a25de70868b581d829bd8e39835b8e2f6a8354a3b2b\'\nSTATIC_SHA = \'513a889477b8940d5f4a1df304b793361ff3860857e4e846e29e28429d9b3d08\'\nMARGIN_BINS = np.asarray([0, .25, .5, 1, 2, 4, np.inf])\nCONFIDENCE_BINS = np.asarray([0, .5, .6, .7, .8, .9, 1.])\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + \'\\n\')\n\n\ndef fixed_confusion(y, pred, nclasses):\n    y, pred = np.asarray(y), np.asarray(pred)\n    assert y.shape == pred.shape and y.ndim == 1\n    assert np.all((y >= 0) & (y < nclasses)) and np.all((pred >= 0) & (pred < nclasses))\n    return np.bincount(y.astype(int) * nclasses + pred.astype(int), minlength=nclasses * nclasses).reshape(nclasses, nclasses)\n\n\ndef rate_interval(values, groups, support, draws=3000, seed=20261018):\n    """Jointly prompt-resample fixed context counts; no fitted uncertainty."""\n    values, support = np.asarray(values), np.asarray(support)\n    assert values.ndim == 2 and support.shape == values.shape[1:]\n    unique, inverse = np.unique(groups, return_inverse=True)\n    count = np.asarray([np.bincount(inverse, weights=row, minlength=len(unique)) for row in values])\n    denom = np.bincount(inverse, weights=support, minlength=len(unique))\n    samples = np.random.default_rng(seed).integers(0, len(unique), size=(draws, len(unique)))\n    total = denom[samples].sum(1)\n    assert np.all(total > 0)\n    rates = count[:, samples].sum(2) / total[None]\n    intervals = np.quantile(rates, [.025, .975], axis=1).T\n    mean = np.quantile(rates.mean(0), [.025, .975])\n    return dict(by_context=intervals.tolist(), mean_across_contexts=mean.tolist(),\n                draws=draws, seed=seed, resampling_unit=\'prompt\',\n                interpretation=\'Conditional descriptive interval for fixed reused-development predictions; omits fitting and new-data uncertainty.\')\n\n\ndef context_overlap(mask, ids):\n    """Count repeated observations separately from the exact unique-row union."""\n    mask, ids = np.asarray(mask, dtype=bool), np.asarray(ids)\n    assert mask.ndim == 2 and mask.shape[1] == len(ids) and len(np.unique(ids)) == len(ids)\n    count = mask.sum(0)\n    patterns = {\'\'.join(str(int(v)) for v in pattern): int(np.all(mask.T == pattern, axis=1).sum())\n                for pattern in np.asarray(np.meshgrid(*([[False, True]] * mask.shape[0]))).T.reshape(-1, mask.shape[0])}\n    return dict(repeated_context_observations=int(mask.sum()), unique_passages_union=int((count > 0).sum()),\n                unique_passages_all_contexts=int((count == mask.shape[0]).sum()),\n                unique_passages_by_number_of_contexts={str(i): int((count == i).sum()) for i in range(mask.shape[0] + 1)},\n                context_pattern_counts=patterns,\n                pairwise_intersections={f\'{SCENARIOS[a]}_{SCENARIOS[b]}\': int((mask[a] & mask[b]).sum())\n                                        for a in range(mask.shape[0]) for b in range(a + 1, mask.shape[0])})\n\n\ndef row_metrics(evidence, probability, y, pair):\n    """Descriptive scores only; no corrective gate or new argmax."""\n    a, b = pair\n    margin = evidence[..., a] - evidence[..., b]\n    # Stable logistic of the ordered pair log-evidence margin.\n    pair_a = np.exp(-np.logaddexp(0, -margin))\n    true_pair = (y == a) | (y == b)\n    orientation = np.where(y == a, 1., np.where(y == b, -1., 0.))\n    ordered = np.argsort(-evidence, axis=2, kind=\'stable\')\n    top2pair = ((ordered[..., :2] == a).any(2) & (ordered[..., :2] == b).any(2))\n    return dict(pair_margin=margin, true_pair=true_pair, orientation=orientation,\n                truth_margin=margin * orientation[None], pair_a_mass=pair_a,\n                pair_confidence=np.maximum(pair_a, 1 - pair_a),\n                global_confidence=probability.max(2), pair_global_mass=probability[..., a] + probability[..., b],\n                top2_is_pair=top2pair, ordered_top2=ordered[..., :2])\n\n\ndef pair_counts(y, pred, pair, top2=None):\n    a, b = pair\n    true_pair = (y == a) | (y == b)\n    pred_pair = (pred == a) | (pred == b)\n    cross = ((y == a) & (pred == b)) | ((y == b) & (pred == a))\n    correct = true_pair & (pred == y)\n    other = true_pair & ~pred_pair\n    false_entry = ~true_pair & pred_pair\n    result = dict(pair_support=int(true_pair.sum()), pair_correct=int(correct.sum()), pair_cross=int(cross.sum()),\n                  pair_other_error=int(other.sum()), a_to_b=int(((y == a) & (pred == b)).sum()),\n                  b_to_a=int(((y == b) & (pred == a)).sum()), false_entries=int(false_entry.sum()),\n                  predicted_pair=int(pred_pair.sum()), global_errors=int((pred != y).sum()))\n    assert result[\'pair_support\'] == result[\'pair_correct\'] + result[\'pair_cross\'] + result[\'pair_other_error\']\n    if top2 is not None:\n        result.update(top2_scope_rows=int(top2.sum()), top2_scope_true_pair=int((top2 & true_pair).sum()),\n                      top2_scope_cross=int((top2 & cross).sum()), top2_scope_pair_correct=int((top2 & correct).sum()),\n                      top2_scope_other_class=int((top2 & ~true_pair).sum()), top2_scope_false_entries=int((top2 & false_entry).sum()))\n    return result\n\n\ndef text_features(texts):\n    """Deterministic numeric passage shape; never return text/tokens."""\n    names = (\'characters\', \'words\', \'lines\', \'mean_word_characters\', \'uppercase_fraction\', \'digit_fraction\',\n             \'punctuation_fraction\', \'blank_line_fraction\', \'heading_line_fraction\', \'bullet_line_fraction\',\n             \'code_fences\', \'questions\')\n    output = np.zeros((len(texts), len(names)), dtype=np.float64)\n    for i, value in enumerate(texts):\n        value = str(value)\n        lines, words = value.splitlines(), value.split()\n        n, nl = max(len(value), 1), max(len(lines), 1)\n        output[i] = (len(value), len(words), len(lines), sum(map(len, words)) / max(len(words), 1),\n                     sum(c.isupper() for c in value) / n, sum(c.isdigit() for c in value) / n,\n                     sum(unicodedata.category(c).startswith(\'P\') for c in value) / n,\n                     sum(not line.strip() for line in lines) / nl,\n                     sum(bool(re.match(r\'^\\s*#{1,6}\\s\', line)) for line in lines) / nl,\n                     sum(bool(re.match(r\'^\\s*(?:[-*+]\\s|\\d+[.)]\\s)\', line)) for line in lines) / nl,\n                     value.count(\'```\'), value.count(\'?\'))\n    return np.asarray(names, dtype=str), output\n\n\ndef _locate(root, filename):\n    paths = list(Path(root).rglob(filename))\n    assert len(paths) == 1, (\'Expected unique input file\', filename, len(paths))\n    return paths[0]\n\n\ndef _load_inputs(root):\n    paths = {name: _locate(root, name) for name in EXPECTED_HASHES}\n    for name, path in paths.items():\n        assert digest(path) == EXPECTED_HASHES[name], (\'Immutable input hash mismatch\', name)\n    result = json.loads(paths[\'ensemble_result.json\'].read_text())\n    manifest = json.loads(paths[\'output_manifest.json\'].read_text())\n    for name in EXPECTED_HASHES:\n        if name not in (\'train.csv\', \'output_manifest.json\'):\n            assert manifest[name][\'sha256\'] == EXPECTED_HASHES[name]\n            assert manifest[name][\'bytes\'] == paths[name].stat().st_size\n    arrays = {}\n    with np.load(paths[\'ensemble_arrays.npz\'], allow_pickle=False) as saved:\n        for key in saved.files:\n            value = saved[key].copy()\n            assert not value.dtype.hasobject and value.dtype.kind in \'biufU\', (\'Unsafe dtype\', key)\n            if value.dtype.kind == \'f\':\n                assert np.isfinite(value).all(), (\'Nonfinite array\', key)\n            arrays[key] = value\n    train = pd.read_csv(paths[\'train.csv\'], dtype={\'id\': str, \'prompt_id\': str, \'model\': str}, keep_default_na=False)\n    return arrays, train, result, paths\n\n\ndef _source_gate(arrays, train, result, paths):\n    ids, classes, y, groups, fold = (arrays[k] for k in (\'ids\', \'classes\', \'y\', \'prompt_ids\', \'fold\'))\n    n, c = len(ids), len(classes)\n    assert n == 2699 and c == 18 and len(np.unique(ids)) == n and len(np.unique(groups)) == 150\n    assert ids.dtype.kind == classes.dtype.kind == groups.dtype.kind == \'U\'\n    assert tuple(arrays[\'scenarios\']) == SCENARIOS and tuple(result[\'scenarios\']) == SCENARIOS\n    assert arrays[\'train_sha256\'].item() == EXPECTED_HASHES[\'train.csv\'] == result[\'train_sha256\']\n    assert result[\'fixed_weights\'] == WEIGHTS\n    assert result[\'canonical_cpu_cache\'][\'status\'] == \'PASS\'\n    assert result[\'canonical_cpu_cache\'][\'canonical\'][\'cache_sha256\'] == CANONICAL_SHA\n    assert result[\'canonical_cpu_cache\'][\'static\'][\'cache_sha256\'] == STATIC_SHA\n    assert result[\'status\'] == \'COMPLETE_CANONICAL_CPU_FIXED_AVAILABLE_COMPONENT_PROXY\'\n    assert result[\'classes\'] == classes.tolist() and np.array_equal(classes, np.sort(train.model.unique()))\n    for key, column in ((\'ids\', \'id\'), (\'prompt_ids\', \'prompt_id\')):\n        np.testing.assert_array_equal(arrays[key], train[column].to_numpy(dtype=str))\n    np.testing.assert_array_equal(classes[y], train.model.to_numpy(dtype=str))\n    expected = {\'p000\': 4, \'p001\': 3, \'p002\': 5}\n    expected.update({f\'p{i:03d}\': 5 - i % 5 for i in range(3, 150)})\n    assert all(expected[g] == int(f) for g, f in zip(groups, fold))\n    for member in MEMBERS:\n        member_fold = arrays[\'fold_\' + member]\n        assert member_fold.shape == (n,) and len(np.unique(member_fold)) == 5\n        for f in np.unique(member_fold):\n            assert len(np.unique(groups[member_fold == f])) == 30\n            assert not set(groups[member_fold == f]) & set(groups[member_fold != f])\n    eligible = arrays[\'eligible\']\n    assert eligible.shape == (3, n) and eligible.dtype.kind == \'b\'\n    base = np.zeros((n, c), dtype=np.float64)\n    terms = []\n    for member in MEMBERS[:5]:\n        probability = arrays[\'probability_\' + member]\n        assert probability.shape == (n, c) and np.all(probability >= 0) and np.all(probability <= 1 + 2e-6)\n        np.testing.assert_allclose(probability.sum(1), 1, rtol=0, atol=2e-6)\n        logs = np.log(np.clip(probability, 1e-9, 1))\n        base += WEIGHTS[member] * logs\n        terms.append(np.broadcast_to(WEIGHTS[member] * logs, (3, n, c)))\n    plain = arrays[\'deb_log_probability_plain\']\n    centered = arrays[\'deb_log_probability_control\']\n    assert plain.shape == centered.shape == (3, n, c)\n    assert np.all(centered[~eligible] == 0)\n    evidence = base[None] + .05 * plain\n    evidence = evidence.copy()\n    evidence[eligible] += .25 * centered[eligible]\n    np.testing.assert_array_equal(evidence, arrays[\'log_evidence_control\'])\n    np.testing.assert_array_equal(evidence.argmax(2), arrays[\'prediction_raw_control\'])\n    terms.extend((.05 * plain, .25 * centered))\n    probability = np.exp(evidence - evidence.max(2, keepdims=True))\n    probability /= probability.sum(2, keepdims=True)\n    np.testing.assert_allclose(probability, arrays[\'raw_probability_control\'], rtol=0, atol=2e-15)\n    scores, confusions = [], []\n    for s, scenario in enumerate(SCENARIOS):\n        cm = fixed_confusion(y, arrays[\'prediction_raw_control\'][s], c)\n        prior = pd.read_csv(paths[f\'raw_control_scenario{scenario}_confusion.csv\'], index_col=0)\n        assert list(prior.index) == list(prior.columns) == classes.tolist()\n        np.testing.assert_array_equal(cm, prior.to_numpy())\n        score = float((cm.diagonal() / cm.sum(1)).mean())\n        np.testing.assert_allclose(score, result[\'scores\'][\'raw_control\'][s], rtol=0, atol=1e-15)\n        scores.append(score); confusions.append(cm)\n    np.testing.assert_allclose(np.mean(scores), result[\'mean_scores\'][\'raw_control\'], rtol=0, atol=1e-15)\n    pair = tuple(int(np.flatnonzero(classes == label)[0]) for label in PAIR)\n    counts = [pair_counts(y, row, pair) for row in arrays[\'prediction_raw_control\']]\n    for key, expected_count in dict(a_to_b=[50,49,52], b_to_a=[44,38,42], pair_correct=[135,146,139],\n                                    pair_other_error=[71,67,67], false_entries=[65,54,49], global_errors=[1278,1206,1174]).items():\n        assert [row[key] for row in counts] == expected_count, (\'Source count reconciliation\', key)\n    contribution = np.stack([term[..., pair[0]] - term[..., pair[1]] for term in terms], axis=2)\n    np.testing.assert_allclose(contribution.sum(2), evidence[..., pair[0]] - evidence[..., pair[1]], rtol=0, atol=1e-12)\n    return dict(status=\'PASS\', immutable_input_hashes=EXPECTED_HASHES, source_script_version_id=359514801,\n                source_canonical_cache_sha256=CANONICAL_SHA, source_static_cache_sha256=STATIC_SHA,\n                ordered_ids_labels_prompts_and_folds_exact=True, exact_raw_evidence_and_predictions=True,\n                source_confusions_exact=True, raw_control_scores=scores,\n                source_raw_mean_balanced_accuracy=float(np.mean(scores)),\n                weighted_pair_contribution_sum_atol=1e-12,\n                raw_probability_replay_atol=2e-15), contribution\n\n\ndef _summaries(values):\n    values = np.asarray(values)\n    if not len(values):\n        return dict(mean=None, median=None, q25=None, q75=None)\n    return dict(mean=float(values.mean()), median=float(np.median(values)),\n                q25=float(np.quantile(values, .25)), q75=float(np.quantile(values, .75)))\n\n\ndef _binned(values, mask, outcome, bins, metric, scenario):\n    rows = []\n    for i in range(len(bins) - 1):\n        active = mask & (values >= bins[i]) & ((values <= bins[i + 1]) if i == len(bins) - 2 else (values < bins[i + 1]))\n        rows.append(dict(scenario=scenario, metric=metric, lower=float(bins[i]),\n                         upper=None if np.isinf(bins[i + 1]) else float(bins[i + 1]),\n                         upper_inclusive=i == len(bins) - 2, rows=int(active.sum()),\n                         correct=int((active & (outcome == 1)).sum()), cross=int((active & (outcome == 2)).sum()),\n                         other_error=int((active & (outcome == 3)).sum())))\n    assert sum(row[\'rows\'] for row in rows) == int(mask.sum())\n    return rows\n\n\ndef analyze(arrays, train, contribution):\n    ids, classes, y, groups, fold = (arrays[k] for k in (\'ids\', \'classes\', \'y\', \'prompt_ids\', \'fold\'))\n    pair = tuple(int(np.flatnonzero(classes == label)[0]) for label in PAIR)\n    evidence, probability, pred = (arrays[k] for k in (\'log_evidence_control\', \'raw_probability_control\', \'prediction_raw_control\'))\n    metrics = row_metrics(evidence, probability, y, pair)\n    true_pair = metrics[\'true_pair\']\n    correct = true_pair[None] & (pred == y[None])\n    cross = ((y[None] == pair[0]) & (pred == pair[1])) | ((y[None] == pair[1]) & (pred == pair[0]))\n    pred_pair = (pred == pair[0]) | (pred == pair[1])\n    false_entry = ~true_pair[None] & pred_pair\n    outcome = np.where(correct, 1, np.where(cross, 2, np.where(true_pair[None], 3, np.where(false_entry, 4, 0)))).astype(np.int8)\n    summaries = [dict(scenario=scenario, **pair_counts(y, pred[s], pair, metrics[\'top2_is_pair\'][s])) for s, scenario in enumerate(SCENARIOS)]\n    bins, member_rows, false_rows, fold_rows, prompt_rows, feature_rows, pair_rows = [], [], [], [], [], [], []\n    feature_names, features = text_features(train.text.to_numpy())\n    orientation = metrics[\'orientation\']\n    truth_contribution = contribution * orientation[None, :, None]\n    votes = np.sign(contribution).astype(np.int8)\n    centered_neutral = ~arrays[\'eligible\']\n    assert np.all(votes[..., -1][centered_neutral] == 0)\n    positive, negative = (votes > 0).sum(2), (votes < 0).sum(2)\n    disagreement = (positive > 0) & (negative > 0)\n    for s, scenario in enumerate(SCENARIOS):\n        for metric, values, edges in ((\'absolute_pair_log_margin\', np.abs(metrics[\'pair_margin\'][s]), MARGIN_BINS),\n                                      (\'global_softmax_max\', metrics[\'global_confidence\'][s], CONFIDENCE_BINS),\n                                      (\'conditional_pair_max\', metrics[\'pair_confidence\'][s], CONFIDENCE_BINS)):\n            bins.extend(_binned(values, true_pair, outcome[s], edges, metric, scenario))\n        for m, member in enumerate(MEMBERS):\n            for code, label in ((1, \'pair_correct\'), (2, \'pair_cross\'), (3, \'pair_other_error\')):\n                selected = outcome[s] == code\n                true_vote = votes[s, :, m] * orientation\n                member_rows.append(dict(scenario=scenario, member=member, outcome=label, rows=int(selected.sum()),\n                                        supports_true=int((selected & (true_vote > 0)).sum()),\n                                        supports_other_pair=int((selected & (true_vote < 0)).sum()),\n                                        neutral=int((selected & (true_vote == 0)).sum()),\n                                        **{\'truth_weighted_margin_\' + k: v for k, v in _summaries(truth_contribution[s, selected, m]).items()}))\n        for cls, label in enumerate(classes):\n            if cls in pair:\n                continue\n            selected = y == cls\n            false_rows.append(dict(scenario=scenario, true_model=str(label), support=int(selected.sum()),\n                                   predicted_a=int((selected & (pred[s] == pair[0])).sum()),\n                                   predicted_b=int((selected & (pred[s] == pair[1])).sum()),\n                                   top2_pair_scope=int((selected & metrics[\'top2_is_pair\'][s]).sum())))\n        for f in range(1, 6):\n            selected = fold == f\n            fold_rows.append(dict(scenario=scenario, fold=f, **pair_counts(y[selected], pred[s, selected], pair, metrics[\'top2_is_pair\'][s, selected])))\n        for group in np.unique(groups):\n            selected = groups == group\n            prompt_rows.append(dict(scenario=scenario, prompt_id=str(group), fold=int(fold[selected][0]),\n                                    **pair_counts(y[selected], pred[s, selected], pair, metrics[\'top2_is_pair\'][s, selected])))\n        for code, label in ((1, \'pair_correct\'), (2, \'pair_cross\'), (3, \'pair_other_error\')):\n            selected = outcome[s] == code\n            for f, name in enumerate(feature_names):\n                feature_rows.append(dict(scenario=scenario, outcome=label, rows=int(selected.sum()), feature=str(name), **_summaries(features[selected, f])))\n        for i in np.flatnonzero(true_pair | false_entry[s]):\n            pair_rows.append(dict(scenario=scenario, id=str(ids[i]), prompt_id=str(groups[i]), fold=int(fold[i]),\n                                  true_model=str(classes[y[i]]), predicted_model=str(classes[pred[s, i]]), is_true_pair=bool(true_pair[i]), outcome=int(outcome[s, i]),\n                                  pair_log_margin=float(metrics[\'pair_margin\'][s, i]), truth_pair_log_margin=float(metrics[\'truth_margin\'][s, i]),\n                                  global_softmax_max=float(metrics[\'global_confidence\'][s, i]), conditional_pair_a_mass=float(metrics[\'pair_a_mass\'][s, i]),\n                                  pair_global_softmax_mass=float(metrics[\'pair_global_mass\'][s, i]), top2_is_pair=bool(metrics[\'top2_is_pair\'][s, i]),\n                                  member_disagreement=bool(disagreement[s, i]), members_supporting_true=int(((votes[s, i] * orientation[i]) > 0).sum()),\n                                  active_centered=bool(arrays[\'eligible\'][s, i])))\n    overlap = {}\n    for name, mask, population in ((\'pair_cross\', cross, true_pair), (\'pair_correct\', correct, true_pair),\n                                   (\'pair_other_error\', outcome == 3, true_pair), (\'nonpair_false_entry\', false_entry, ~true_pair)):\n        overlap[name] = context_overlap(mask[:, population], ids[population])\n        overlap[name][\'population_passages\'] = int(population.sum())\n        overlap[name][\'population\'] = \'true_pair\' if name != \'nonpair_false_entry\' else \'true_nonpair\'\n    repeated = int(cross.sum()); union = overlap[\'pair_cross\'][\'unique_passages_union\']\n    assert repeated >= union\n    uncertainty = dict(pair_correct_rate=rate_interval(correct, groups, true_pair), pair_cross_rate=rate_interval(cross, groups, true_pair))\n    disagreements = [dict(scenario=scenario, pair_rows_with_member_disagreement=int((true_pair & disagreement[s]).sum()),\n                          cross_rows_with_member_disagreement=int((cross[s] & disagreement[s]).sum()),\n                          cross_rows_with_any_member_supporting_true=int((cross[s] & ((votes[s] * orientation[:, None]) > 0).any(1)).sum()),\n                          cross_rows_with_no_member_supporting_true=int((cross[s] & ~((votes[s] * orientation[:, None]) > 0).any(1)).sum()),\n                          inactive_centered_rows=int((~arrays[\'eligible\'][s]).sum())) for s, scenario in enumerate(SCENARIOS)]\n    saved = dict(ids=ids, classes=classes, y=y, prompt_ids=groups, fold=fold, scenarios=np.asarray(SCENARIOS),\n                 pair_class_indices=np.asarray(pair), source_prediction_raw_control=pred,\n                 pair_true_mask=true_pair, pair_outcome=outcome, pair_margin=metrics[\'pair_margin\'],\n                 truth_pair_margin=metrics[\'truth_margin\'], pair_a_mass=metrics[\'pair_a_mass\'],\n                 pair_global_mass=metrics[\'pair_global_mass\'], global_softmax_max=metrics[\'global_confidence\'],\n                 top2_is_pair=metrics[\'top2_is_pair\'], ordered_top2=metrics[\'ordered_top2\'], members=np.asarray(MEMBERS),\n                 weighted_pair_contribution=contribution, member_pair_preference=votes, member_disagreement=disagreement,\n                 centered_active=arrays[\'eligible\'], text_feature_names=feature_names, text_features=features)\n    return dict(summary=summaries, overlap=overlap, uncertainty=uncertainty, disagreements=disagreements), saved, dict(\n        confidence_bins=bins, component_contributions=member_rows, other_class_false_entries=false_rows, folds=fold_rows,\n        prompts=prompt_rows, text_features=feature_rows, pair_rows=pair_rows)\n\n\ndef run_diagnostics(input_root, output_root, freeze_path=None):\n    output_root = Path(output_root)\n    assert not output_root.exists() or not any(output_root.iterdir()), \'Fresh diagnostic output directory required\'\n    output_root.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    def status(value, **extra):\n        write_json(output_root / \'status.json\', dict(status=value, observed_utc=datetime.now(timezone.utc).isoformat(), **extra))\n    status(\'VERIFYING_IMMUTABLE_SOURCE\')\n    try:\n        freeze_receipt = {}\n        if freeze_path is not None:\n            freeze = json.loads(Path(freeze_path).read_text())\n            assert digest(__file__) == freeze[\'diagnostic_source_sha256\']\n            plan_path = Path(freeze.get(\'plan_path\', Path(__file__).with_name(\'STAGE_06F_PAIR_DIAGNOSTICS.md\')))\n            assert digest(plan_path) == freeze[\'plan_sha256\']\n            assert freeze[\'input_version_ids\'] == [170484, 359514801]\n            assert freeze[\'accelerator\'] == \'none\' and freeze[\'internet_enabled\'] is False\n            freeze_receipt = dict(freeze_sha256=digest(freeze_path), diagnostic_source_sha256=digest(__file__), plan_sha256=digest(plan_path), input_version_ids=freeze[\'input_version_ids\'])\n        arrays, train, source_result, paths = _load_inputs(input_root)\n        gate, contribution = _source_gate(arrays, train, source_result, paths)\n        gate[\'execution_freeze\'] = freeze_receipt\n        write_json(output_root / \'source_gate.json\', gate)\n        print(\'STAGE06F_SOURCE_GATE\', json.dumps(gate, allow_nan=False), flush=True)\n        status(\'DESCRIPTIVE_FIXED_PAIR_DIAGNOSIS\')\n        findings, saved, tables = analyze(arrays, train, contribution)\n        assert all(not value.dtype.hasobject and value.dtype.kind in \'biufU\' for value in saved.values())\n        np.savez_compressed(output_root / \'pair_diagnostics.npz\', **saved)\n        with np.load(output_root / \'pair_diagnostics.npz\', allow_pickle=False) as readback:\n            assert set(readback.files) == set(saved)\n            for key, value in saved.items():\n                np.testing.assert_array_equal(readback[key], value)\n        for name, rows in tables.items():\n            pd.DataFrame(rows).to_csv(output_root / (name + \'.csv\'), index=False)\n        result = dict(status=\'COMPLETE_FIXED_PAIR_DIAGNOSTICS\', pair=list(PAIR), source_script_version_id=359514801,\n                      primary_source=\'raw_control\', contexts=list(SCENARIOS), rows=len(arrays[\'ids\']),\n                      distinct_true_pair_passages=int(saved[\'pair_true_mask\'].sum()), context_rows_are_repeated_passages=True,\n                      source_raw_scores=gate[\'raw_control_scores\'], source_raw_mean=gate[\'source_raw_mean_balanced_accuracy\'],\n                      **findings, source_gate=gate, runtime=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__),\n                      fits=0, model_inference_calls=0, regrouping_calls=0, optimization_or_threshold_search=False,\n                      changed_predictions=0, corrective_candidate_evaluated=False, official_submission=False, paid_services=False,\n                      seconds=time.monotonic()-started,\n                      limitations=[\'Available-component development proxy, not full S02 or an official score.\',\n                                   \'Pair selected after inspecting reused-development confusion counts; findings generate hypotheses only.\',\n                                   \'Contexts reuse the same passages and trained heads; repeated counts are not independent examples.\',\n                                   \'Softmax and conditional pair masses are descriptive uncalibrated scores.\',\n                                   \'Prompt bootstrap fixes all learned models and predictions; it omits fitting, grouping and future-data uncertainty.\',\n                                   \'Text-shape associations and component disagreement do not establish causal mechanisms or recoverable accuracy.\',\n                                   \'Top-two scope is diagnostic only and never changes a prediction.\'])\n        write_json(output_root / \'pair_result.json\', result)\n        manifest = {p.name: dict(bytes=p.stat().st_size, sha256=digest(p)) for p in sorted(output_root.iterdir())\n                    if p.is_file() and p.name not in (\'status.json\', \'output_manifest.json\')}\n        write_json(output_root / \'output_manifest.json\', manifest)\n        status(\'COMPLETE_FIXED_PAIR_DIAGNOSTICS\', valid=True)\n        for name in (\'component_contributions\', \'confidence_bins\', \'other_class_false_entries\', \'folds\', \'text_features\'):\n            print(\'STAGE06F_\' + name.upper() + \'_CSV\', json.dumps(pd.DataFrame(tables[name]).to_csv(index=False)), flush=True)\n        print(\'STAGE06F_RESULT\', json.dumps(result, allow_nan=False), flush=True)\n        print(\'STAGE06F_OUTPUT_MANIFEST\', json.dumps(manifest, allow_nan=False), flush=True)\n        print(\'STAGE06F_COMPLETE\', flush=True)\n        return result\n    except BaseException as error:\n        status(\'FAILED_PRESERVE_OUTPUTS\', exception_type=type(error).__name__, exception_message=str(error))\n        print(\'STAGE06F_FAILED\', type(error).__name__, str(error), flush=True)\n        raise\n\n\nif __name__ == \'__main__\':\n    freeze_path = os.environ.get(\'NTL_STAGE06F_FREEZE\')\n    assert freeze_path, \'Production execution requires NTL_STAGE06F_FREEZE\'\n    run_diagnostics(os.environ.get(\'NTL_STAGE06F_INPUT\', \'/kaggle/input\'),\n                    os.environ.get(\'NTL_STAGE06F_OUTPUT\', \'/kaggle/working/stage06f_diagnostics\'), freeze_path)\n', 'STAGE_06F_PAIR_DIAGNOSTICS.md': '# Stage 06F: fixed Gemini pair diagnosis from saved Stage 06E evidence\n\nStatus: prospective, diagnostic only. This plan precedes the diagnostic execution. There is no new fitted or corrective model and no candidate score.\n\n## Scope and immutable inputs\n\nUse only the completed private Stage 06E V2, scriptVersionId `359514801`, and official training data. Require the exact source `ensemble_arrays.npz` SHA256 `9348eeed149ad48bdd828d8055e945f57e384d8917c654125c2ebbe8d4b99519`, source result SHA256 `9717bcb12f0c730b4af631067621969dcdbf239e6bed2f84304b470a38458018`, and official train SHA256 `40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707`. The E result binds canonical cache `2c6aa02a0bafd9389f436a25de70868b581d829bd8e39835b8e2f6a8354a3b2b` and static-member cache `513a889477b8940d5f4a1df304b793361ff3860857e4e846e29e28429d9b3d08`; neither cache needs loading because its consumed numeric evidence is already saved in the bound ensemble arrays.\n\nFreeze `gemini-3.7-flash` versus `gemini-3.8-flash` as the sole target pair. Primary diagnostics use the raw control ensemble in contexts `[1,3,6]`. The same 2,699 passages recur in every context. These are repeated predictions, not three independent samples. Do not use the rejected centering candidate or batch balancing to choose corrections.\n\nBefore any diagnostic table, verify hashes, numeric/Unicode-only NPZ schema with `allow_pickle=False`, official IDs/classes/labels/prompt alignment, exact archived prompt fold mapping and saved component fold prompt exclusion. Replay the original ordered seven-member evidence arithmetic and require exact raw evidence and argmax equality. Verify saved softmax at tight tolerance; reconcile all three complete raw-control confusion matrices, scores, class supports and known pair cross-errors `[94,87,94]`, directed counts `[50,49,52]` and `[44,38,42]`. Failure preserves fresh-output status and blocks completion. Record actual Python/NumPy/pandas runtime rather than selecting an environment by the observed result.\n\n## Fixed descriptive outputs\n\n1. Per context: pair supports, correct/cross/other errors, true pair examples predicted outside the pair, non-pair false entries, and the diagnostic scope where the raw top two labels are exactly the pair. No gate changes a prediction.\n2. Row IDs and three-context outcome patterns: exact counts of unique passages with cross-errors in one, two or three contexts, intersections and union. Repeated context count remains separately labelled. Prompt and fold tables preserve the context coordinate.\n3. Signed log-evidence margin `L(3.7)-L(3.8)`, label-oriented margin for true-pair rows, raw maximum softmax mass, and pair-conditional mass `exp(L(3.7))/(exp(L(3.7))+exp(L(3.8)))`. These are descriptive scores; they are not calibrated correctness probabilities. Fixed absolute-margin bins `[0,.25,.5,1,2,4,infinity]`; fixed global/pair-confidence bins `[0,.5,.6,.7,.8,.9,1]`. The final bin includes the upper endpoint.\n4. Each of the five static members plus DeBERTa plain/centered: fixed-weight signed pair contribution, truth-oriented contribution summaries for pair-correct/cross/other rows, within-pair preference and member disagreement. Zero differences and inactive centered contributions are neutral, never first-class tie votes. The contribution decomposition must sum to the raw pair margin within `atol=1e-12`; exact replay uses the original fusion order instead.\n5. Pair false entries by true non-pair class, top-two diagnostic scope, prompt/fold concentration and deterministic passage-shape features: character/word/line counts, mean word length, uppercase/digit/punctuation fractions, blank/heading/bullet-line fractions, code-fence and question counts. Persist only numeric summaries and row identifiers; never passage text, snippets, tokens or transcripts. Associations do not establish a mechanism.\n6. Conditional descriptive 95% intervals for pair-correct and pair-cross rates by context and their mean: 3,000 prompt-cluster bootstrap draws with seed `20261018`; jointly resample the original 150 prompts across contexts, keep all stored predictions fixed, and use the same 300 true-pair passages repeatedly. These are not refit or untouched-validation uncertainty intervals. No multiple-comparison significance or future-performance claim is made.\n\nSave numeric/Unicode `pair_diagnostics.npz`, aggregate CSVs/JSON, pair-row CSV without text, source gate, output manifest and terminal status. Print compact aggregate machine-readable tables, result and source audit for independent extraction. Record zero fits/inference/regrouping/search and no changed predictions. Require a fresh output directory. Root packaging freezes source/plan hashes and verifies saved CPU/offline source and pinned inputs before the single planned diagnostic execution.\n\n## Interpretation and next decision\n\nThe aim is to identify falsifiable hypotheses: persistent versus context-sensitive errors, existing members that disagree with incorrect pair preference, and whether pair-directed scope exposes material other-class risk. Do not select a specialist model, threshold, feature, weight or calibration using a newly optimized score. Any corrective recipe requires its own prospective plan and prompt-held-out full raw ensemble comparison, including all classes and false entries. The available-component proxy and reused development errors do not constitute full S02 evidence or untouched generalization. Official S02 remains separate at 54.9%; this diagnostic cannot increase it.\n\nNo model fitting, classifier invocation, affine readout, feature embeddings, grouping, calibration, threshold sweep, paid service, GPU, submission, final-selection change or messages to others are authorized by this stage.\n', 'EXECUTION_FREEZE.json': '{\n  "schema_version": "stage06f-execution-freeze-v1",\n  "status": "FROZEN_SOURCE_CANDIDATE_NOT_EXECUTED",\n  "diagnostic_source_sha256": "b264d2f7398fda6127c306057c2e41f3d170cd430bd1d115189f7ddf40fa8d64",\n  "plan_sha256": "a62a1f2e70c061b9b5c6b4be1825f3f81e148a0a3b6c37f36d12899fb8432d29",\n  "source_path": "/kaggle/working/stage06f_sources/stage06f_pair_diagnostics.py",\n  "plan_path": "/kaggle/working/stage06f_sources/STAGE_06F_PAIR_DIAGNOSTICS.md",\n  "input_version_ids": [\n    170484,\n    359514801\n  ],\n  "stage06e_script_version_id": 359514801,\n  "expected_source_input_sha256": {\n    "ensemble_arrays.npz": "9348eeed149ad48bdd828d8055e945f57e384d8917c654125c2ebbe8d4b99519",\n    "ensemble_result.json": "9717bcb12f0c730b4af631067621969dcdbf239e6bed2f84304b470a38458018",\n    "train.csv": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",\n    "output_manifest.json": "626b428135f4ff377d593de29c716042a07593c74678893f874c91b7b71e2bbf",\n    "raw_control_scenario1_confusion.csv": "a48b61f2a7ffb50764d44bb060adb925e13122abd63fed4730c9ba097685c595",\n    "raw_control_scenario3_confusion.csv": "fae2720b0f34b0bb701bfdffaace99051f0add84d52c91e6973aa6f124299463",\n    "raw_control_scenario6_confusion.csv": "938f3bf5a1ea51fdda8b8a75512810c5c74002063e741feef0d8e5dd614c7513"\n  },\n  "accelerator": "none",\n  "internet_enabled": false,\n  "expected_thread_limits": {\n    "OMP_NUM_THREADS": "2",\n    "OPENBLAS_NUM_THREADS": "2",\n    "MKL_NUM_THREADS": "2"\n  },\n  "planned_model_fits": 0,\n  "planned_model_inference_calls": 0,\n  "planned_changed_predictions": 0,\n  "official_submission_performed": false,\n  "limitation": "Preparation only. Saved private Kaggle source, pinned inputs, CPU/offline settings and independent review must be verified before execution."\n}\n'}
STAGE06F_FILE_SHA256 = {'stage06f_pair_diagnostics.py': 'b264d2f7398fda6127c306057c2e41f3d170cd430bd1d115189f7ddf40fa8d64', 'STAGE_06F_PAIR_DIAGNOSTICS.md': 'a62a1f2e70c061b9b5c6b4be1825f3f81e148a0a3b6c37f36d12899fb8432d29', 'EXECUTION_FREEZE.json': 'd50aaec1ff33f475fce4438fff032533de18196fa9ce1e8df560eeb6f77f8c94'}


In [ ]:
# Execute one diagnostic only after all frozen source bytes match.
try:
    assert all(not root.exists() for root in STAGE06F_ROOTS.values()), (
        'Fresh Stage 06F source and output roots required; preserve any prior trial',
        [str(root) for root in STAGE06F_ROOTS.values() if root.exists()])
    STAGE06F_ROOTS['pipeline'].mkdir(parents=True, exist_ok=False)
    STAGE06F_PIPELINE_CREATED = True
    stage06f_pipeline_status('MATERIALIZING_REVIEWED_SOURCES')
    STAGE06F_ROOTS['sources'].mkdir(parents=True, exist_ok=False)
    for name, source in STAGE06F_SOURCE_TEXT.items():
        assert Path(name).name == name
        encoded = source.encode('utf-8')
        assert hashlib.sha256(encoded).hexdigest() == STAGE06F_FILE_SHA256[name]
        path = STAGE06F_ROOTS['sources'] / name
        with path.open('xb') as stream:
            stream.write(encoded)
        assert path.read_bytes() == encoded
        assert hashlib.sha256(path.read_bytes()).hexdigest() == STAGE06F_FILE_SHA256[name]
    assert {path.name for path in STAGE06F_ROOTS['sources'].iterdir()} == set(STAGE06F_SOURCE_TEXT)
    stage06f_pipeline_json('materialized_sources.json', dict(
        files=STAGE06F_FILE_SHA256, exact_text_bytes_verified=True,
        input_version_ids=STAGE06F_INPUT_VERSION_IDS,
        accelerator='none', internet_enabled=False,
        source_verification_scope='Runtime local materialization; saved private notebook and attached versions require separate verification.'))
    os.environ.update({
        'NTL_STAGE06F_INPUT': '/kaggle/input',
        'NTL_STAGE06F_OUTPUT': str(STAGE06F_ROOTS['diagnostics']),
        'NTL_STAGE06F_FREEZE': str(STAGE06F_ROOTS['sources'] / 'EXECUTION_FREEZE.json'),
    })
    assert all(os.environ[key] == '2' for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'))
    stage06f_pipeline_status('VERIFY_AND_DIAGNOSE_SAVED_EVIDENCE')
    runpy.run_path(str(STAGE06F_ROOTS['sources'] / 'stage06f_pair_diagnostics.py'), run_name='__main__')
    stage06f_terminal = json.loads((STAGE06F_ROOTS['diagnostics'] / 'status.json').read_text())
    stage06f_result = json.loads((STAGE06F_ROOTS['diagnostics'] / 'pair_result.json').read_text())
    assert stage06f_terminal['status'] == 'COMPLETE_FIXED_PAIR_DIAGNOSTICS' and stage06f_terminal['valid'] is True
    assert stage06f_result['status'] == 'COMPLETE_FIXED_PAIR_DIAGNOSTICS'
    assert stage06f_result['source_script_version_id'] == 359514801 and stage06f_result['primary_source'] == 'raw_control'
    assert all(stage06f_result[key] == 0 for key in ('fits', 'model_inference_calls', 'regrouping_calls', 'changed_predictions'))
    assert stage06f_result['corrective_candidate_evaluated'] is False and stage06f_result['official_submission'] is False
    assert (STAGE06F_ROOTS['diagnostics'] / 'output_manifest.json').is_file()
    stage06f_pipeline_status('COMPLETE_FIXED_PAIR_DIAGNOSTICS',
        model_fits=0, model_inference_calls=0, changed_predictions=0,
        official_submission_performed=False)
    print('STAGE06F_PIPELINE_COMPLETE', flush=True)
except BaseException as stage06f_error:
    failure = dict(status='FAILED_PRESERVE_OUTPUTS', failed_phase=STAGE06F_PHASE,
        exception_type=type(stage06f_error).__name__, exception_message=str(stage06f_error),
        traceback=traceback.format_exc(), observed_utc=datetime.now(timezone.utc).isoformat(),
        elapsed_seconds=time.monotonic() - STAGE06F_STARTED,
        completed_diagnostic=False, official_submission_performed=False,
        source_sha256=STAGE06F_FILE_SHA256)
    if STAGE06F_PIPELINE_CREATED:
        stage06f_pipeline_json('failure.json', failure)
        stage06f_pipeline_json('status.json', failure)
    print('STAGE06F_PIPELINE_FAILED', json.dumps(failure, allow_nan=False), flush=True)
    raise
